In [1]:
import pandas as pd

hill = pd.read_csv("../data/hillstrom.csv")
print(hill.shape)
hill.head()

(64000, 12)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


In [2]:
hill["segment"].value_counts()

segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

In [3]:
hill.groupby("segment")["visit"].mean() * 100

segment
Mens E-Mail      18.275684
No E-Mail        10.616728
Womens E-Mail    15.140038
Name: visit, dtype: float64

In [4]:
from scipy.stats import chi2_contingency

table = pd.crosstab(hill["segment"], hill["visit"])
chi2, p, dof, expected = chi2_contingency(table)
print(table)
print("p-value:", p)

visit              0     1
segment                   
Mens E-Mail    17413  3894
No E-Mail      19044  2262
Womens E-Mail  18149  3238
p-value: 2.869178384267155e-110


In [5]:
import numpy as np

p_mens = 3894 / 21307
p_none = 2262 / 21306
diff = p_mens - p_none
se = np.sqrt(p_mens * (1 - p_mens) / 21307 + p_none * (1 - p_none) / 21306)

print("lift:", round(diff * 100, 1), "points")
print("95% CI:", round((diff - 1.96 * se) * 100, 1), "to", round((diff + 1.96 * se) * 100, 1), "points")

lift: 7.7 points
95% CI: 7.0 to 8.3 points


## A/B test result

Customers were randomly split into three equal groups (~21,300 each). Visit rates were 10.6% with no email, 15.1% with the Women's email and 18.3% with the Men's email. A chi-square test gave p ≈ 0, so I reject the null hypothesis: the emails caused more visits. The Men's email raised visits by about 7.7 percentage points (95% CI roughly 7.0 to 8.3).

In [6]:
X = pd.get_dummies(hill[["recency", "history", "mens", "womens", "newbie", "zip_code", "channel"]], dtype=int)
X.head()

,recency,history,mens,womens,newbie,zip_code_Rural,zip_code_Surburban,zip_code_Urban,channel_Multichannel,channel_Phone,channel_Web
0,10,142.44,1,0,0,0,1,0,0,1,0
1,6,329.08,1,1,1,1,0,0,0,0,1
2,7,180.65,0,1,1,0,1,0,0,0,1
3,9,675.83,1,0,1,1,0,0,0,0,1
4,2,45.34,1,0,0,0,0,1,0,0,1


split into learning (70%) and testing (30%)

In [7]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(hill, test_size=0.3, random_state=42, stratify=hill["segment"])
X_train = X.loc[train.index]
X_test = X.loc[test.index]

train one model per group

In [8]:
from lightgbm import LGBMClassifier

models = {}
for group in ["No E-Mail", "Mens E-Mail", "Womens E-Mail"]:
    rows = train["segment"] == group
    m = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
    m.fit(X_train[rows], train.loc[rows, "visit"])
    models[group] = m

uplift for each test customer, and their best email

In [9]:
p_none = models["No E-Mail"].predict_proba(X_test)[:, 1]
test["uplift_mens"] = models["Mens E-Mail"].predict_proba(X_test)[:, 1] - p_none
test["uplift_womens"] = models["Womens E-Mail"].predict_proba(X_test)[:, 1] - p_none

test["best_email"] = np.where(test["uplift_mens"] >= test["uplift_womens"], "Mens E-Mail", "Womens E-Mail")
test["best_uplift"] = test[["uplift_mens", "uplift_womens"]].max(axis=1)

test[["uplift_mens", "uplift_womens", "best_email", "best_uplift"]].head(10)

,uplift_mens,uplift_womens,best_email,best_uplift
28167,0.045876,0.026124,Mens E-Mail,0.045876
10061,0.056633,0.020204,Mens E-Mail,0.056633
34681,0.202363,0.019953,Mens E-Mail,0.202363
5877,0.101437,0.050982,Mens E-Mail,0.101437
26326,0.165146,0.246624,Womens E-Mail,0.246624
9901,0.079042,-0.020125,Mens E-Mail,0.079042
35101,0.197683,-0.050563,Mens E-Mail,0.197683
50908,0.087085,0.019567,Mens E-Mail,0.087085
6257,0.165820,0.108960,Mens E-Mail,0.165820
41797,0.061293,0.070039,Womens E-Mail,0.070039


In [10]:
test["uplift_group"] = pd.qcut(test["uplift_mens"], 5, labels=["1 lowest", "2", "3", "4", "5 highest"])

check = test[test["segment"].isin(["Mens E-Mail", "No E-Mail"])]
result = check.groupby(["uplift_group", "segment"], observed=True)["visit"].mean().unstack() * 100
result["real_lift"] = result["Mens E-Mail"] - result["No E-Mail"]
result.round(1)

segment,Mens E-Mail,No E-Mail,real_lift
uplift_group,,,
1 lowest,16.7,11.8,4.9
2,16.6,9.0,7.6
3,15.5,8.3,7.2
4,18.4,9.7,8.6
5 highest,22.1,12.5,9.6


## Uplift model

I trained one LightGBM model per group (no email, Men's email, Women's email) to predict each customer's chance of visiting, and took the difference as their uplift (the T-learner approach). Each customer gets the email with the highest predicted uplift.

To check it on the test set, I split customers into 5 groups by predicted uplift and compared their real visit rates. The real lift from the Men's email rose from +4.9 points in the lowest group to +9.6 points in the highest. So the model does find the customers who respond most, although the differences between groups are moderate.